In [1]:
!pip install transformers datasets torch
!pip install evaluate
import pandas as pd
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.5 M

In [2]:
#Dataset Credits: Lei Huang
import pandas as pd
from datasets import Dataset

In [3]:
# Load your dataset
df = pd.read_csv('sentiment_yelp_data.csv')
df.head()

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny,sentiment,sentiment_label
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,wife took birthday breakfast excellent weather...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0,Positive,2
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,idea people give bad reviews place goes show p...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0,Positive,2
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love gyro plate rice good also dig candy selec...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0,Positive,2
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,rosie dakota love chaparral dog park convenien...,review,uZetl9T0NcROGOyFfughhg,1,2,0,Positive,2
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,general manager scott petello good egg go deta...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0,Positive,2


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   business_id      10000 non-null  object
 1   date             10000 non-null  object
 2   review_id        10000 non-null  object
 3   stars            10000 non-null  int64 
 4   text             9999 non-null   object
 5   type             10000 non-null  object
 6   user_id          10000 non-null  object
 7   cool             10000 non-null  int64 
 8   useful           10000 non-null  int64 
 9   funny            10000 non-null  int64 
 10  sentiment        10000 non-null  object
 11  sentiment_label  10000 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 937.6+ KB


In [5]:
df.sentiment.unique()

array(['Positive', 'Negative', 'Neutral'], dtype=object)

In [6]:
# Check for any missing or non-string values in the 'text' column
print(df['text'].isnull().sum())  # Check for null values
print(df['text'].apply(lambda x: isinstance(x, str)).sum())  # Check for non-string entries


1
9999


In [7]:
# Drop rows where 'text' is null
df = df.dropna(subset=['text'])

# Ensure all values in the 'text' column are strings
df['text'] = df['text'].astype(str)


In [8]:
# Convert the dataset to a Hugging Face `Dataset`
# Ensure the `sentiment` column is mapped to integers (0 for negative, 1 for positive)
df['sentiment'] = df['sentiment'].map({'Negative': 0, 'Neutral': 1, 'Positive': 2})
dataset = Dataset.from_pandas(df)

# Peek at the dataset to ensure it's loaded correctly
print(dataset)

Dataset({
    features: ['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id', 'cool', 'useful', 'funny', 'sentiment', 'sentiment_label', '__index_level_0__'],
    num_rows: 9999
})


In [9]:
#Tokenize data using BERT Tokenizer

In [10]:
from transformers import BertTokenizer

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize function to apply to each example
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Apply the tokenizer to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/9999 [00:00<?, ? examples/s]

In [11]:
#Prepare data for PyTorch Training

In [12]:
# Remove unnecessary columns
tokenized_dataset = tokenized_dataset.remove_columns(['business_id', 'date','review_id', 'stars', 'type', 'user_id', 'cool', 'useful', 'funny', 'sentiment_label'])

# Rename the sentiment_label column to labels
tokenized_dataset = tokenized_dataset.rename_column('sentiment', 'labels')

# Set the dataset format to PyTorch tensors
tokenized_dataset.set_format('torch')



In [13]:
#Split dataset to train and test


In [14]:
# Split the dataset into training and testing sets (80% train, 20% test)
train_test_split = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print(train_dataset)
print(test_dataset)


Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 7999
})
Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


In [15]:
train_dataset['labels']

tensor([2, 2, 2,  ..., 2, 2, 2])

In [16]:
#Load the model

In [17]:
from transformers import BertForSequenceClassification

# Load pre-trained BERT for sequence classification (with 3 sentiment labels)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
#Define training parameters and initialize trainer
from evaluate import load
# Load evaluation metrics using the evaluate library
accuracy_metric = load("accuracy")
precision_metric = load("precision")
recall_metric = load("recall")
f1_metric = load("f1")

# Define a function to compute evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)

    # Compute precision, recall, f1 for each label
    precision = precision_metric.compute(predictions=predictions, references=labels, average=None)
    recall = recall_metric.compute(predictions=predictions, references=labels, average=None)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average=None)

    # Compute macro averages for sensitivity analysis
    macro_precision = np.mean(precision['precision'])
    macro_recall = np.mean(recall['recall'])
    macro_f1 = np.mean(f1['f1'])

    return {
        "accuracy": accuracy["accuracy"],
        "precision_negative": precision['precision'][0],
        "precision_neutral": precision['precision'][1],
        "precision_positive": precision['precision'][2],
        "recall_negative": recall['recall'][0],
        "recall_neutral": recall['recall'][1],
        "recall_positive": recall['recall'][2],
        "f1_negative": f1['f1'][0],
        "f1_neutral": f1['f1'][1],
        "f1_positive": f1['f1'][2],
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
    }





In [19]:
from transformers import Trainer, TrainingArguments

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory to save model
    evaluation_strategy="epoch",     # evaluate each epoch
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    num_train_epochs=15,              # number of epochs change 1 to 15
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for logs
    logging_steps=10,
)

# Set up training arguments
#training_args = TrainingArguments(
#    output_dir="./results",
#    evaluation_strategy="epoch",
#    per_device_train_batch_size=8,
#    per_device_eval_batch_size=8,
#    num_train_epochs=3,
#    weight_decay=0.01,
#    logging_dir="./logs",
#    logging_steps=10,
#    load_best_model_at_end=True,
#    metric_for_best_model="macro_f1",  # Use Macro F1 score to select the best model
#    greater_is_better=True,
#)


# Initialize the trainer
trainer = Trainer(
    model=model,                     # the pre-trained BERT model
    args=training_args,              # training arguments
    train_dataset=train_dataset,     # training dataset
    eval_dataset=test_dataset,        # evaluation dataset
    compute_metrics=compute_metrics,
)

# Initialize the Trainer
#trainer = Trainer(
#    model=model,
#    args=training_args,
#    train_dataset=tokenized_datasets["train"],
#    eval_dataset=tokenized_datasets["validation"],
#    compute_metrics=compute_metrics,
#)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [20]:
#Train Model / Fine-Tune BERT
trainer

In [21]:
# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision Negative,Precision Neutral,Precision Positive,Recall Negative,Recall Neutral,Recall Positive,F1 Negative,F1 Neutral,F1 Positive,Macro Precision,Macro Recall,Macro F1
1,0.233900,0.283344,0.907000,0.519126,0.000000,0.946065,0.508021,0.000000,0.962486,0.513514,0.000000,0.954205,0.488397,0.490169,0.489239
2,0.294300,0.351432,0.915500,0.608333,0.000000,0.935106,0.390374,0.000000,0.984323,0.475570,0.000000,0.959083,0.514480,0.458232,0.478218
3,0.232900,0.343585,0.898500,0.480176,0.000000,0.952059,0.582888,0.000000,0.945129,0.526570,0.000000,0.948581,0.477412,0.509339,0.491717
4,0.260600,0.285361,0.918500,0.600000,0.000000,0.950000,0.577540,0.000000,0.968085,0.588556,0.000000,0.958957,0.516667,0.515208,0.515838
5,0.232500,0.384138,0.909500,0.661765,0.000000,0.918219,0.240642,0.000000,0.993281,0.352941,0.000000,0.954276,0.526661,0.411308,0.435739
6,0.401100,0.384277,0.910000,0.759259,0.000000,0.914183,0.219251,0.000000,0.996081,0.340249,0.000000,0.953376,0.557814,0.405111,0.431208
7,0.144300,0.377520,0.922000,0.812500,0.750000,0.927301,0.347594,0.222222,0.992721,0.486891,0.342857,0.958897,0.829934,0.520846,0.596215
8,0.218900,0.315017,0.922500,0.662252,0.500000,0.948606,0.534759,0.370370,0.971445,0.591716,0.425532,0.959889,0.703619,0.625525,0.659046
9,0.137200,0.333984,0.922500,0.700000,0.666667,0.938437,0.449198,0.296296,0.981523,0.547231,0.410256,0.959496,0.768368,0.575672,0.638995
10,0.091200,0.357507,0.925000,0.696296,0.500000,0.945376,0.502674,0.296296,0.978723,0.583851,0.372093,0.961761,0.713891,0.592564,0.639235


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning

TrainOutput(global_step=7500, training_loss=0.18364953443109988, metrics={'train_runtime': 2545.7008, 'train_samples_per_second': 47.132, 'train_steps_per_second': 2.946, 'total_flos': 3.156966342609408e+16, 'train_loss': 0.18364953443109988, 'epoch': 15.0})

In [22]:
#Evaluate Model

In [23]:
# Evaluate the model
results = trainer.evaluate()
print(results)


{'eval_loss': 0.402058482170105, 'eval_accuracy': 0.924, 'eval_precision_negative': 0.7, 'eval_precision_neutral': 0.5185185185185185, 'eval_precision_positive': 0.9457406402604449, 'eval_recall_negative': 0.48663101604278075, 'eval_recall_neutral': 0.5185185185185185, 'eval_recall_positive': 0.9759238521836506, 'eval_f1_negative': 0.5741324921135648, 'eval_f1_neutral': 0.5185185185185185, 'eval_f1_positive': 0.9605952052907137, 'eval_macro_precision': 0.7214197195929878, 'eval_macro_recall': 0.66035779558165, 'eval_macro_f1': 0.684415405307599, 'eval_runtime': 12.4223, 'eval_samples_per_second': 161.001, 'eval_steps_per_second': 10.063, 'epoch': 15.0}


In [24]:
#Save Model

In [25]:
# Save the model and tokenizer
model.save_pretrained('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')


('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/special_tokens_map.json',
 './fine_tuned_model/vocab.txt',
 './fine_tuned_model/added_tokens.json')

In [26]:
!zip mymodel.zip fine_tuned_model/

  adding: fine_tuned_model/ (stored 0%)


In [27]:
#Using SavedModel

In [28]:
from transformers import BertTokenizer, BertForSequenceClassification

# Load the saved tokenizer
tokenizer = BertTokenizer.from_pretrained('./fine_tuned_model')

# Load the saved model
model = BertForSequenceClassification.from_pretrained('./fine_tuned_model')

# Example text to classify sentiment
text = "The movie was amazing and I loved it!"

# Tokenize the input text (same as how it was done during training)
inputs = tokenizer(text, return_tensors='pt', padding='max_length', truncation=True)

import torch

# Perform inference (get the logits)
outputs = model(**inputs)

# Extract the predicted label (index of the maximum value in logits)
predictions = torch.argmax(outputs.logits, dim=1)

# Map the prediction to the actual label (e.g., positive, negative, neutral)
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}  # Adjust according to your label mapping
predicted_label = label_map[predictions.item()]

print(f"Predicted sentiment: {predicted_label}")


Predicted sentiment: Positive


In [29]:
tokenizer

BertTokenizer(name_or_path='./fine_tuned_model', vocab_size=30522, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [30]:
inputs
print (type(inputs))

<class 'transformers.tokenization_utils_base.BatchEncoding'>
